# Heterogeneous Data Ingestion Patterns — Main Lab

Run the **Data Setup** notebook first if inputs are missing.


## Learning Objectives

1) Explain design and architecture considerations.
2) Report on business usefulness.
3) Demonstrate combining CSV, logs, and API JSON.

## 0) Imports & Configuration

In [1]:
from pathlib import Path
from urllib.parse import parse_qs
import re, json
import pandas as pd
import requests
DATA_DIR = Path('./data')
print('Data dir:', DATA_DIR.resolve())

Data dir: /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data


## 0.1) Check inputs

In [2]:
req = ['orders.csv','access.log','users_fallback.json']
missing = [p for p in req if not (DATA_DIR/p).exists()]
if missing:
    raise FileNotFoundError(f'Missing files: {missing}. Run Data Setup first.')
print('All inputs found.')

All inputs found.


## 1) CSV — Orders

In [5]:
orders_path = DATA_DIR / 'orders.csv'
df_orders_raw = pd.read_csv(orders_path, dtype={'order_id':'string','user_id':'int64'})
print(len(df_orders_raw))
df_orders_raw.head()

10000


,order_id,user_id,order_total,order_ts
0,O1,90,51.45,2025-08-16T23:55:00
1,O2,774,135.96,2025-08-17T19:23:00
2,O3,654,155.70,2025-08-21T09:37:00
3,O4,439,41.66,2025-08-16T02:16:00
4,O5,433,18.33,2025-08-16T03:14:00


## 2) Logs — Parse access.log

In [6]:
log_path = DATA_DIR / 'access.log'
print('Preview:\n', '\n'.join(log_path.read_text().splitlines()[:3]))

Preview:
 127.0.0.1 - - [2025-08-22T15:40:09] "GET /cart?uid=625 HTTP/1.1" 404 2460 "-" "Mozilla/5.0"
127.0.0.1 - - [2025-08-22T14:39:14] "GET /home?uid=833 HTTP/1.1" 200 1445 "-" "Mozilla/5.0"
127.0.0.1 - - [2025-08-22T11:42:37] "GET /checkout?uid=873 HTTP/1.1" 200 2174 "-" "Mozilla/5.0"


In [ ]:
pattern = re.compile(
    r'^(?P<ip>\S+) \S+ \S+ \[(?P<ts>[^\]]+)\] ' 
    r'"(?P<method>\S+) (?P<path>[^?\s"]+)(?:\?(?P<query>[^"]*))? HTTP/(?P<httpver>\d\.\d)" '
    r'(?P<status>\d{3}) (?P<bytes>\d+) "[^"]*" "[^"]*"$'
)
# breakdown
# ^(?P<ip>\S+) captures named group "ip" for the first set of characters before a whitespace
# \S+ matches any length of string between whitespaces (x2) (note the spaces between)
# \[(?P<ts>[^\]]+)\] captures everything between the two square brackets and names it "ts"
# (?P<method>\S+) captures the method e.g. GET and names it "method" (note the " before it)
# (?P<path>[^?\s"]+) captures any string before a ?, space or " and names it "path"
# (?:\?(?P<query>[^"]*))? captures the query between the ? and the " and names it "query"
# (?P<status>\d{3}) captures the three-digit status code and names it "status"
# (?P<bytes>\d+) caputures a digit-string and names it "bytes"
# The rest of the data is pulled through

In [16]:
rows, bad = [], []
with open(DATA_DIR / "access.log", "r") as f:
    for line in f:
        line = line.rstrip("\n")
        m = pattern.match(line)
        if not m:
            bad.append(line)
            continue
        g = m.groupdict()
        # Parse uid from the query string (if present)
        q = g.get("query") or ""
        uid_vals = parse_qs(q, keep_blank_values=True).get("uid", [])
        uid = int(uid_vals[0]) if uid_vals and uid_vals[0].isdigit() else None

        rows.append({
            "ts": pd.to_datetime(g["ts"], errors="coerce"),
            "path": g["path"],
            "status": int(g["status"]),
            "bytes": int(g["bytes"]),
            "user_id": uid
        })

df_logs = (
    pd.DataFrame(rows, columns=["ts","path","status","bytes","user_id"])
      .dropna(subset=["ts"])  # optional: drop unparsable timestamps
      .sort_values("ts")
      .reset_index(drop=True)
)

print("Parsed log rows:", len(df_logs), "| Bad lines:", len(bad))

Parsed log rows: 5000 | Bad lines: 0


In [17]:
df_logs.head()

,ts,path,status,bytes,user_id
0,2025-08-22 10:00:00,/checkout,302,3253,768
1,2025-08-22 10:00:11,/home,200,850,41
2,2025-08-22 10:00:13,/,404,2751,731
3,2025-08-22 10:00:14,/product/42,404,3863,265
4,2025-08-22 10:00:19,/product/7,200,2190,697


## 3) API JSON — With Fallback

In [18]:
def load_users(base_dir: Path = DATA_DIR) -> pd.DataFrame:
    url = 'https://jsonplaceholder.typicode.com/users'
    try:
        r = requests.get(url, timeout=10); r.raise_for_status()
        data, src = r.json(), 'api'
    except Exception as e:
        print('API fetch failed — using fallback. Reason:', repr(e))
        data, src = json.loads((base_dir/'users_fallback.json').read_text()), 'fallback'
    df = pd.json_normalize(data); df['source'] = src; return df

In [41]:
df_users_raw = load_users()
print(df_users_raw.shape)
df_users_raw[['id','name','email','address.city','source']].head()

(10, 16)


,id,name,email,address.city,source
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,api
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,api
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,api
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,api
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,api


## 4) Validate & Deduplicate

In [20]:
assert df_orders_raw['user_id'].isna().sum()==0, 'Null user_id in orders!'

In [22]:
df_orders_raw['order_ts'] = pd.to_datetime(df_orders_raw['order_ts'], errors='coerce')

In [27]:
df_orders = df_orders_raw\
        .sort_values('order_ts')\
        .drop_duplicates(subset=['order_id'], keep='first')\
        .reset_index(drop=True)
print(len(df_orders_raw))
print(len(df_orders))

10000
9999


In [29]:
df_orders_raw.groupby(['order_id']).count()

,user_id,order_total,order_ts
order_id,,,
O1,2,2,2
O10,1,1,1
O100,1,1,1
O1000,1,1,1
O10000,1,1,1
...,...,...,...
O9995,1,1,1
O9996,1,1,1
O9997,1,1,1


In [30]:
assert df_users_raw['id'].isna().sum()==0, 'Null id in users!'

In [32]:
df_users = df_users_raw.rename(columns={'id':'user_id','address.city':'city'}).copy()
df_users = df_users[['user_id','name','email','city','source']]
df_users.head()

,user_id,name,email,city,source
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,api
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,api
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,api
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,api
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,api


## 5) Aggregate & Combine

In [34]:
activity = df_logs\
        .groupby('user_id')\
        .agg(pageviews=('user_id','size'), last_seen=('ts','max'))\
        .reset_index()
activity.head()

,user_id,pageviews,last_seen
0,1,4,2025-08-22 15:55:00
1,2,1,2025-08-22 15:30:29
2,3,7,2025-08-22 15:31:54
3,4,4,2025-08-22 15:33:44
4,5,1,2025-08-22 15:36:26


In [36]:
orders_summary = df_orders\
        .groupby('user_id')\
        .agg(total_orders=('order_id','nunique'), total_revenue=('order_total','sum'))\
        .reset_index()
orders_summary.head()

,user_id,total_orders,total_revenue
0,1,9,1047.69
1,2,10,1070.51
2,3,5,553.01
3,4,10,996.67
4,5,18,2038.77


In [39]:
combined = df_users\
    .merge(orders_summary, on='user_id', how='left')\
    .merge(activity, on='user_id', how='left')\
    .sort_values(['total_revenue', 'pageviews'], ascending=[False, False])\
    .reset_index(drop=True)

In [40]:
combined_view = combined[['user_id','name','email','city','total_orders','total_revenue','pageviews','last_seen','source']]
combined_view = combined_view.fillna({'total_orders':0,'total_revenue':0,'pageviews':0})
combined_view.head(10)

,user_id,name,email,city,total_orders,total_revenue,pageviews,last_seen,source
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,18,2038.77,1,2025-08-22 15:36:26,api
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me,Aliyaview,14,1442.78,4,2025-08-22 13:41:29,api
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy,9,1079.15,7,2025-08-22 14:22:16,api
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,10,1070.51,1,2025-08-22 15:30:29,api
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,9,1047.69,4,2025-08-22 15:55:00,api
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,10,996.67,4,2025-08-22 15:33:44,api
9,10,Clementina DuBuque,Rey.Padberg@karina.biz,Lebsackbury,7,978.87,6,2025-08-22 14:46:10,api
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz,Howemouth,10,929.40,3,2025-08-22 13:59:38,api
8,9,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury,8,817.42,5,2025-08-22 15:39:31,api
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,5,553.01,7,2025-08-22 15:31:54,api


## 6) Save Output

In [42]:
out = DATA_DIR / 'combined.csv'
combined.to_csv(out, index=False)
print('Wrote:', out.resolve())

Wrote: /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data/combined.csv


## 7) Reflection

Write 3–4 sentences describing a decision this dataset enables.